# P3 — Collections, arrays and thinking in whole series

**Python primer · session 3 of 4**


## What this session is for

This is the session that decides whether your project runs in seconds or in
minutes.

A second of LIGO data is 4096 numbers. A thirty-two second stretch is a hundred
and thirty thousand. A search over a day is three hundred and fifty million.
Nothing you do to that data can be done one number at a time.

So the session has two halves. First the ordinary Python containers — lists,
tuples and dictionaries — which hold small collections of mixed things. Then
**arrays**, which hold large collections of numbers and which you operate on
whole. The second half is the one that matters.


## Lists: an ordered collection you can change


In [ ]:
detectors = ["H1", "L1", "V1"]

print(detectors[0])          # first item -- counting starts at zero
print(detectors[-1])         # last item
print(len(detectors))        # how many

detectors.append("K1")       # add to the end
print(detectors)


Indexing from zero means the last item is at index `len - 1` and `-1` is a
shorthand for it. **Slicing** takes a piece:


In [ ]:
numbers = [10, 20, 30, 40, 50, 60]

print(numbers[1:4])     # from 1 up to but not including 4
print(numbers[:3])      # from the start
print(numbers[3:])      # to the end
print(numbers[::2])     # every second one


`start:stop` always **excludes** the stop. It looks arbitrary until you notice
that `numbers[:3]` and `numbers[3:]` together give you everything exactly once,
with no overlap and no gap.


### Changing a list

A list is **mutable**: it can be changed in place and the methods that do it
are worth knowing by name.


In [ ]:
events = ["GW150914", "GW151226", "GW170817"]

events.append("GW190521")            # add at the end
events.insert(1, "GW151012")         # add at a position
events.remove("GW151226")            # remove by value
last = events.pop()                  # remove and return the last

print(events)
print("popped:", last)
print("sorted:", sorted(events))     # a NEW sorted list
events.sort()                        # sorts in place, returns nothing
print("in place:", events)


> **Careful — Two habits that look the same and are not**
>
> `sorted(x)` returns a new sorted list and leaves `x` alone. `x.sort()` sorts
> `x` itself and returns `None`.
>
> Writing `x = x.sort()` therefore replaces your list with nothing and it is a
> mistake everybody makes once. The same distinction runs through Python:
> methods that change an object in place usually return `None`.

Membership and length work on any collection:


In [ ]:
print("GW170817" in events)
print(len(events))
for i, name in enumerate(events):        # index and value together
    print(i, name)


### Sets: a collection without duplicates

A **set** holds unordered unique values. Use one when the question is
*membership* rather than order.


In [ ]:
detected_h1 = {"GW150914", "GW151226", "GW170817"}
detected_v1 = {"GW170817", "GW170814"}

print(detected_h1 & detected_v1)     # in both -- a coincident detection
print(detected_h1 | detected_v1)     # in either
print(detected_h1 - detected_v1)     # in H1 only
print(len({"a", "b", "a"}))          # duplicates collapse


The intersection of two sets of triggers is, in miniature, what a coincidence
search does: keep what two detectors saw at the same time and throw away the
rest.


### Nesting

Containers hold containers and real data is always nested.


In [ ]:
catalogue = {
    "GW150914": {"m1": 36.0, "m2": 29.0, "detectors": ["H1", "L1"]},
    "GW170817": {"m1": 1.46, "m2": 1.27, "detectors": ["H1", "L1", "V1"]},
}

for name, event in catalogue.items():
    total = event["m1"] + event["m2"]
    n = len(event["detectors"])
    print(f"{name}: {total:5.1f} solar masses, seen by {n} detectors")

print(catalogue["GW170817"]["detectors"][2])


Read `catalogue["GW170817"]["detectors"][2]` from the left: take the entry for
that event, then its list of detectors, then the third one. Every JSON file you
open in the next session has exactly this shape.


## Tuples and dictionaries

A **tuple** is a list you cannot change, written with round brackets. Use one
when the number of items is fixed and meaningful — a pair of masses, a
frequency band.


In [ ]:
band = (35.0, 350.0)          # the band we filter GW150914 in, in hertz
low, high = band              # unpacking, one name per item
print(low, high)


A **dictionary** stores values under names rather than positions.


In [ ]:
event = {
    "name": "GW150914",
    "gps": 1126259462.4,
    "m1": 36.0,
    "m2": 29.0,
    "distance_mpc": 410.0,
}

print(event["name"])
print(event["m1"] + event["m2"], "solar masses in total")

event["snr"] = 24.4           # add a new entry
print(list(event.keys()))


Dictionaries are how almost every scientific file format hands you its
contents: an HDF5 file, a JSON response from an event catalogue, the metadata
of a strain file. Next session opens one.


## Arrays: the object this course runs on

A NumPy array holds many numbers **of the same type** and lets you operate on
all of them at once.


In [ ]:
import numpy as np

a = np.array([1.0, 2.0, 3.0, 4.0])
print(a)
print(a.shape)      # how many, in each dimension
print(a.dtype)      # what kind of number
print(a * 2)        # every element, doubled -- no loop


That last line is the whole idea. `a * 2` did not multiply a list by two; it
multiplied **every element** by two and gave back a new array.

Ways to make one:


In [ ]:
print(np.zeros(5))
print(np.arange(0, 10, 2))            # start, stop (excluded), step
print(np.linspace(0, 1, 5))           # start, stop (included), how many
rng = np.random.default_rng(42)         # seeded: same numbers every run
print(rng.standard_normal(4))           # gaussian noise


Note the `default_rng(42)`: the number is a **seed** and it makes the random
numbers the same every time you run the cell. Any figure in this course that
uses random numbers seeds them, because a plot that changes on every run cannot
be checked by anybody.


### Why not a loop

Here is the same job done twice on a realistic amount of data: thirty-two
seconds of strain, 131 072 samples.


In [ ]:
import time

n = 4096 * 32                       # thirty-two seconds of strain
data = np.random.default_rng(0).standard_normal(n)
print(n, "samples")


First the way that feels natural — one sample at a time.


In [ ]:
out = np.empty(n)

t0 = time.perf_counter()
for i in range(n):
    out[i] = data[i] * 2
t_loop = time.perf_counter() - t0

print(f"loop:  {t_loop * 1000:8.2f} ms")


Now the whole array at once.


In [ ]:
t0 = time.perf_counter()
out2 = data * 2
t_vector = time.perf_counter() - t0

print(f"array: {t_vector * 1000:8.2f} ms")
print(f"speedup: {t_loop / t_vector:.0f} times")
print("same answer:", np.array_equal(out, out2))


Tens of times faster, on thirty-two seconds of one detector. A search runs
over months of data from three detectors.

The reason is not that Python is slow at arithmetic. It is that a Python loop
spends its time on bookkeeping — checking types, wrapping and unwrapping
objects — once per element, while `data * 2` hands the whole block to compiled
code that does the arithmetic and nothing else.

> **Important — The habit to build this session**
>
> Whenever you catch yourself writing `for i in range(len(...))` over data, stop
> and ask what the whole-array version is. There almost always is one and it is
> usually shorter as well as faster.
>
> This is what people mean by **vectorised thinking** and it is the one habit
> that separates code that survives a real dataset from code that does not.


### Arrays go through functions too

Any NumPy function applied to an array works element by element:


In [ ]:
t = np.linspace(0, 1, 5)
print(np.sin(2 * np.pi * t))
print(np.sqrt(np.array([1.0, 4.0, 9.0])))
print(np.abs(np.array([-1.0, 2.0, -3.0])))


And two arrays of the same shape combine element by element, which is how a
signal gets added to noise:


In [ ]:
rng = np.random.default_rng(1)
t = np.linspace(0, 0.2, 2000)
signal = 1e-21 * np.sin(2 * np.pi * 100 * t)
noise = 3e-21 * rng.standard_normal(t.size)
measured = signal + noise

print(measured.shape, measured.dtype)
print(f"signal rms {signal.std():.2e}, noise rms {noise.std():.2e}")


## Slicing and selecting

Arrays slice like lists and they also select by **condition**, which lists
cannot do.


In [ ]:
print(measured[:5])          # the first five samples
print(measured[-5:])         # the last five

loud = measured > 5e-21      # an array of True and False, same shape
print(loud[:10])
print("samples above 5e-21:", loud.sum())          # True counts as 1
print("their mean value:", measured[loud].mean())


`measured[loud]` keeps only the samples where the condition is `True`. This is
a **boolean mask** and it is how every selection in this course is written:
events above a threshold, times inside a window, frequencies inside a band.


### Reductions: many numbers to one


In [ ]:
print(f"mean     {measured.mean():.3e}")
print(f"std      {measured.std():.3e}")
print(f"max      {measured.max():.3e}")
print(f"argmax   {measured.argmax()}   <- the index, not the value")
print(f"time of the maximum: {t[measured.argmax()]:.4f} s")


`argmax` gives the **position** of the largest value, which you then use to
look up the time it happened. That pair — find the index, then index the time
array with it — is how every peak in this course is located.


## Exercises


### 1. From a loop to an array

Rewrite this loop as a single array expression and check that the two agree.

```python
result = []
for x in range(1000):
    result.append(x ** 2)
```


In [ ]:
# your code here


> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


### 2. Find the loudest sample

Using the `measured` array from earlier, find the largest absolute value, the
index where it occurs and the time. Then count how many samples exceed three
times the standard deviation.


In [ ]:
# your code here


> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


### 3. A band of frequencies

Make a frequency array from 0 to 1000 Hz with 4096 points and use a boolean
mask to select the band from 35 to 350 Hz. How many points are in the band and
what fraction of the total is that?


In [ ]:
# your code here


> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


## Before the next session

**Do.** `reference/modulo_4_data_structures/exercises/`, with solutions beside
them.

**Check.** You should be able to say why `data * 2` beats a loop, write a
boolean mask and explain what `argmax` returns.

**Next session** is the payoff: a real strain file from the disk, opened
and plotted as the instrument recorded it.
